In [ ]:

import os
import time
import json
import requests
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score
from dotenv import load_dotenv
import joblib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
"""
ETH WHALE ALPHA PIPELINE - COMPLETE INTEGRATED SOLUTION WITH ALL FIXES APPLIED
All 6 phases implemented with proper data flow
Fixed per requirements:
✅ FIX A - Hardened data leakage
✅ FIX B - Remove veto dominance  
✅ FIX C - Structural vs flow vs context vetoes
✅ FIX D - Fix R3 short properly
✅ FIX E - Prevent confidence saturation
✅ FIX F - Final signal object

NEW CRITICAL FIXES IMPLEMENTED:
✅ FIX 1: Price-not-near-lows requirement for shorts
✅ FIX 2: Flow required for all shorts
✅ FIX 3: Redefined R3 short philosophy (early weakness only)
✅ FIX 4: R3 confidence cap at 0.70
✅ FIX 5: R5 stronger flow requirement (flow_score >= 2)

✅ MULTI-KEY ROTATION SYSTEM: Added to prevent rate limiting
"""

import os
import time
import json
import warnings
import requests
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from dotenv import load_dotenv
import joblib

warnings.filterwarnings('ignore')
load_dotenv()

# ========== CONFIGURATION ==========
# Multi-Key Rotation System for Dune API
DUNE_API_KEYS = [
    os.getenv("DUNE_WHALES_API"),
    os.getenv("DUNE_LASEVEN7"),
    os.getenv("DUNE_FIRSTBML"),
    os.getenv("DUNE_LASEVEN71"),
    os.getenv("DUNE_LASEVEN7_TEAM"),
    os.getenv("DUNE_FIRSTBML_TEAM")
]

# Filter out None or empty keys
DUNE_API_KEYS = [key for key in DUNE_API_KEYS if key and str(key).strip()]
if not DUNE_API_KEYS:
    raise ValueError("No Dune API keys found in environment variables!")

print(f"🔑 Loaded {len(DUNE_API_KEYS)} Dune API keys for rotation")

COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY")

# Create directories
for d in ['data', 'data/price_cache', 'validation', 'backtest', 'models', 'logs']:
    os.makedirs(d, exist_ok=True)

# API endpoints
QUERIES = {
    "whales": ("6395391", "data/dune_whales_cache.json", "data/whale_ml_ready.csv"),
    "market_intent": ("6385600", "data/dune_intent_cache.json", "data/market_intent_ml_ready.csv")
}

# Date ranges
COINGECKO_START = pd.Timestamp('2017-05-01', tz='UTC')
DUNE_START = pd.Timestamp('2017-10-16', tz='UTC')

# Feature sets
SHORT_FEATURES = [
    'exchange_flow_share', 'net_exchange_flow_ratio', 'whale_exchange_flow_ratio',
    'whale_exchange_asymmetry', 'eth_vol7', 'eth_vol30', 'btc_ret_lag1',
    'btc_ret_lag3', 'eth_btc_corr_30d', 'whale_volume_ratio_delta_3d',
    'exchange_volume_zscore'
]

LONG_FEATURES = [
    'btc_rsi', 'eth_vol7', 'whale_volume_ratio', 'eth_rsi',
    'btc_ret_lag1', 'eth_burned_zscore_90d', 'eth_btc_corr_30d',
    'eth_ret_lag1', 'btc_ret_lag7', 'btc_vol30',
    'whale_volume_ratio_delta_1d', 'whale_volume_ratio_delta_3d',
    'exchange_flow_share', 'net_exchange_flow_ratio',
    'whale_exchange_flow_ratio', 'tx_per_active_zscore_90d'
]

# Trading parameters
SLIPPAGE = 0.0008
FEES = 0.0004

# Veto classifications
STRUCTURAL_VETOES = ["btc_breakdown", "vol_expansion"]
FLOW_VETOES = ["whale_to_exchange", "net_flow_negative_with_liquidity"]
CONTEXT_VETOES = ["btc_conflict", "low_volatility"]

# ========== MULTI-KEY ROTATION SYSTEM ==========
class DuneKeyRotator:
    """Manages rotation between multiple Dune API keys to avoid rate limits"""
    
    def __init__(self, api_keys):
        self.api_keys = api_keys
        self.key_index = 0
        self.key_usage = {key: {"count": 0, "last_used": None, "errors": 0} for key in api_keys}
        self.total_requests = 0
        self.rate_limit_log = []
        
    def get_next_key(self):
        """Get the next API key using round-robin with error weighting"""
        if not self.api_keys:
            raise ValueError("No API keys available")
        
        # Try to find a key with lowest error rate and recent usage
        available_keys = []
        for key in self.api_keys:
            usage = self.key_usage[key]
            
            # Skip keys with too many recent errors
            if usage["errors"] >= 3:
                continue
                
            # Give preference to keys not used recently
            if usage["last_used"]:
                time_since_use = (datetime.now() - usage["last_used"]).total_seconds()
                if time_since_use < 60:  # 1 minute cooldown
                    continue
                    
            available_keys.append((key, usage["count"], usage["errors"]))
        
        if not available_keys:
            # If all keys have issues, use the one with least errors
            available_keys = [(key, self.key_usage[key]["count"], self.key_usage[key]["errors"]) 
                            for key in self.api_keys]
        
        # Sort by error count (ascending), then usage count (ascending)
        available_keys.sort(key=lambda x: (x[2], x[1]))
        selected_key = available_keys[0][0]
        
        # Update usage stats
        self.key_usage[selected_key]["count"] += 1
        self.key_usage[selected_key]["last_used"] = datetime.now()
        self.key_index = self.api_keys.index(selected_key)
        self.total_requests += 1
        
        # Log usage
        if self.total_requests % 10 == 0:
            self.log_key_usage()
            
        return selected_key
    
    def mark_error(self, key):
        """Mark a key as having an error"""
        if key in self.key_usage:
            self.key_usage[key]["errors"] += 1
            print(f"⚠️  Key error count: {key[:8]}... now has {self.key_usage[key]['errors']} errors")
    
    def mark_success(self, key):
        """Mark a key as successful (reset error count after successful use)"""
        if key in self.key_usage and self.key_usage[key]["errors"] > 0:
            self.key_usage[key]["errors"] = max(0, self.key_usage[key]["errors"] - 1)
    
    def log_key_usage(self):
        """Log current key usage statistics"""
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "total_requests": self.total_requests,
            "key_usage": {key[:8]: stats for key, stats in self.key_usage.items()}
        }
        self.rate_limit_log.append(log_entry)
        
        # Keep only last 1000 log entries
        if len(self.rate_limit_log) > 1000:
            self.rate_limit_log = self.rate_limit_log[-1000:]
        
        # Save to file occasionally
        if len(self.rate_limit_log) % 100 == 0:
            with open("logs/key_rotation_log.json", "w") as f:
                json.dump(self.rate_limit_log[-100:], f, indent=2)
    
    def get_stats(self):
        """Get current rotation statistics"""
        return {
            "total_keys": len(self.api_keys),
            "total_requests": self.total_requests,
            "active_keys": sum(1 for key, stats in self.key_usage.items() if stats["errors"] < 3),
            "key_details": {key[:8]: stats for key, stats in self.key_usage.items()}
        }

# Initialize the key rotator
key_rotator = DuneKeyRotator(DUNE_API_KEYS)

# ========== ENHANCED DUNE API CLIENT ==========
class DuneAPIClient:
    """Enhanced Dune API client with retry logic and key rotation"""
    
    def __init__(self, key_rotator):
        self.key_rotator = key_rotator
        self.session = requests.Session()
        self.session.headers.update({
            "Content-Type": "application/json",
            "Accept": "application/json"
        })
    
    def make_request(self, method, url, **kwargs):
        """Make a request with retry logic and key rotation"""
        max_retries = 3
        base_delay = 1  # seconds
        
        for attempt in range(max_retries):
            # Get API key for this attempt
            api_key = self.key_rotator.get_next_key()
            headers = kwargs.get('headers', {}).copy()
            headers["x-dune-api-key"] = api_key
            kwargs['headers'] = headers
            
            try:
                response = self.session.request(method, url, **kwargs)
                
                if response.status_code == 429:  # Rate limited
                    print(f"⏳ Rate limited on key {api_key[:8]}..., attempt {attempt + 1}/{max_retries}")
                    self.key_rotator.mark_error(api_key)
                    
                    # Exponential backoff
                    delay = base_delay * (2 ** attempt)
                    print(f"   Waiting {delay} seconds before retry...")
                    time.sleep(delay)
                    continue
                    
                elif response.status_code >= 500:  # Server error
                    print(f"⚠️  Server error {response.status_code} on key {api_key[:8]}...")
                    self.key_rotator.mark_error(api_key)
                    
                    delay = base_delay * (2 ** attempt)
                    time.sleep(delay)
                    continue
                    
                elif response.status_code == 200:
                    self.key_rotator.mark_success(api_key)
                    return response
                    
                else:
                    # Other errors
                    print(f"❌ API error {response.status_code}: {response.text[:200]}")
                    self.key_rotator.mark_error(api_key)
                    
                    if attempt == max_retries - 1:
                        return response
                        
                    delay = base_delay * (2 ** attempt)
                    time.sleep(delay)
                    
            except requests.exceptions.Timeout:
                print(f"⏱️  Timeout on attempt {attempt + 1}/{max_retries}")
                self.key_rotator.mark_error(api_key)
                
                if attempt == max_retries - 1:
                    raise
                    
                delay = base_delay * (2 ** attempt)
                time.sleep(delay)
                
            except requests.exceptions.RequestException as e:
                print(f"🌐 Network error: {e}")
                self.key_rotator.mark_error(api_key)
                
                if attempt == max_retries - 1:
                    raise
                    
                delay = base_delay * (2 ** attempt)
                time.sleep(delay)
        
        # If all retries failed
        raise Exception(f"Failed after {max_retries} attempts")
    
    def get(self, url, **kwargs):
        return self.make_request("GET", url, **kwargs)
    
    def post(self, url, **kwargs):
        return self.make_request("POST", url, **kwargs)

# Initialize the enhanced API client
dune_client = DuneAPIClient(key_rotator)

# ========== PRICE NOT NEAR LOWS HELPER ==========
def price_not_near_lows(row, df, lookback=90, min_pct=0.25):
    """
    ✅ FIX 1: Require price to be above X percentile of recent range
    Prevents shorts from triggering at bottoms
    """
    if pd.isna(row['eth_price']):
        return False
    
    # Get the index of the current row
    idx = row.name
    
    # Get recent price window
    start_idx = max(0, idx - lookback)
    recent_prices = df.iloc[start_idx:idx]['eth_price'].values
    
    if len(recent_prices) < 10:
        return True  # Not enough data, allow trade
    
    # Calculate price percentile in recent range
    price_min = np.min(recent_prices)
    price_max = np.max(recent_prices)
    
    if price_max - price_min < 1e-9:  # Flat range
        return True
    
    pct = (row['eth_price'] - price_min) / (price_max - price_min)
    return pct >= min_pct

# ========== FIX A: SAFE ROLLING FUNCTIONS ==========
def rolling_zscore_safe(series, window=90):
    """FIXED: Shift AFTER calculation to prevent leakage"""
    return ((series - series.rolling(window).mean()) / 
            series.rolling(window).std()).shift(1)

def rolling_feature_safe(series, window, func='mean'):
    """Safe rolling with shift"""
    if func == 'mean':
        return series.rolling(window).mean().shift(1)
    elif func == 'std':
        return series.rolling(window).std().shift(1)
    elif func == 'median':
        return series.rolling(window).median().shift(1)
    return series

# ========== POSITION SIZING (FIX F) ==========
def map_confidence_to_size(conf):
    """Position sizing based on confidence"""
    if conf < 0.55: return 0.0
    if conf < 0.60: return 0.25
    if conf < 0.65: return 0.50
    if conf < 0.70: return 0.75
    if conf < 0.75: return 1.00
    if conf < 0.80: return 1.25
    return 1.50

# ========== DATA FETCHING FUNCTIONS ==========
def to_utc(ts):
    """Ensure timestamp is UTC"""
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")

def test_dune_connection():
    """Test Dune API connection with key rotation"""
    print("\n" + "="*70)
    print("TESTING DUNE API CONNECTION WITH KEY ROTATION")
    print("="*70)
    
    stats = key_rotator.get_stats()
    print(f"🔑 Loaded {stats['total_keys']} API keys")
    print(f"   Active keys: {stats['active_keys']}")
    
    # Test each key
    for i, key in enumerate(DUNE_API_KEYS):
        print(f"\nTesting key {i+1}/{len(DUNE_API_KEYS)}: {key[:8]}...")
        
        headers = {"x-dune-api-key": key}
        
        try:
            # Test with a simple query status check
            query_url = f"https://api.dune.com/api/v1/query/{QUERIES['whales'][0]}"
            resp = requests.get(query_url, headers=headers, timeout=10)
            
            if resp.status_code == 200:
                print(f"   ✅ Key {key[:8]}... is valid")
                key_rotator.mark_success(key)
            else:
                print(f"   ❌ Key {key[:8]}... failed: {resp.status_code}")
                key_rotator.mark_error(key)
                
        except Exception as e:
            print(f"   ❌ Key {key[:8]}... error: {e}")
            key_rotator.mark_error(key)
    
    # Check if we have at least one working key
    active_keys = sum(1 for key, stats in key_rotator.key_usage.items() if stats["errors"] < 3)
    
    if active_keys == 0:
        print("\n❌ No working API keys found!")
        return False
    
    print(f"\n✅ {active_keys}/{len(DUNE_API_KEYS)} API keys are working")
    return True

def fetch_dune_incremental(qid, cache_path, query_name="whale_data"):
    """
    Truly incremental Dune data fetching with date-range queries
    Now uses key rotation and enhanced API client
    """
    today = pd.Timestamp.now(tz='UTC').normalize()
    yesterday = today - pd.Timedelta(days=1)
    
    print(f"\n📊 Fetching {query_name}...")
    print(f"   Using Multi-Key Rotation System with {len(DUNE_API_KEYS)} keys")
    
    # Load existing cache if it exists
    df_cached = pd.DataFrame()
    last_date = None
    
    if os.path.exists(cache_path):
        try:
            with open(cache_path) as f:
                cached = json.load(f)
            
            print(f"   Cache file loaded, size: {len(cached.get('data', []))} rows")
            
            if 'data' in cached and cached['data']:
                df_cached = pd.DataFrame(cached["data"])
                # Check what columns are in the cached data
                print(f"   Cached columns: {list(df_cached.columns)}")
                
                if 'block_date' in df_cached.columns:
                    df_cached["block_date"] = pd.to_datetime(df_cached["block_date"], utc=True)
                    last_date = df_cached["block_date"].max()
                    
                    if last_date >= yesterday:
                        print(f"✅ {query_name} cache current ({last_date.date()})")
                        return df_cached
                    
                    print(f"📅 Cache exists up to {last_date.date()}, fetching new data...")
                else:
                    print(f"⚠️  Cache exists but missing 'block_date' column")
                    last_date = DUNE_START
            else:
                print(f"⚠️  Cache exists but empty, fetching from scratch...")
                last_date = DUNE_START
                
        except (json.JSONDecodeError, KeyError, ValueError) as e:
            print(f"⚠️  Cache corrupted, recreating: {e}")
            last_date = DUNE_START
    else:
        print(f"📝 No cache found, fetching from {DUNE_START.date()}...")
        last_date = DUNE_START
    
    # ✅ FIX: Ensure we start from DUNE_START when cache is empty
    # Check if we're doing the initial fetch
    if df_cached.empty:
        print(f"🔄 Initial fetch detected, starting from {DUNE_START.date()}")
        fetch_start = DUNE_START
    else:
        # Determine fetch range - start from day after last date
        fetch_start = last_date + pd.Timedelta(days=1)
    
    fetch_end = yesterday
    
    # Check if we need to fetch anything
    if fetch_start > fetch_end:
        print(f"✅ No new data needed (fetch_start: {fetch_start.date()}, fetch_end: {fetch_end.date()})")
        return df_cached
    
    print(f"🔍 Fetching range: {fetch_start.date()} to {fetch_end.date()}")
    
    # ✅ FIX: Add buffer for initial fetch to ensure we get earliest data
    if df_cached.empty and fetch_start == DUNE_START:
        print(f"   Initial fetch - requesting data from the earliest available date")
    
    # Prepare query parameters
    query_params = {
        "start_date": fetch_start.strftime("%Y-%m-%d"),
        "end_date": fetch_end.strftime("%Y-%m-%d")
    }
    
    try:
        # Execute query with parameters using enhanced client
        execute_url = f"https://api.dune.com/api/v1/query/{qid}/execute"
        execute_payload = {
            "query_parameters": query_params
        }
        
        print(f"🚀 Executing Dune query {qid} with params: {query_params}")
        print(f"   Using Multi-Key Rotation System")
        
        resp = dune_client.post(
            execute_url,
            json=execute_payload,
            timeout=60
        )
        
        print(f"   Response status: {resp.status_code}")
        
        if resp.status_code != 200:
            print(f"❌ API error: {resp.status_code} - {resp.text[:200]}")
            # Try to use cached data if API fails
            if not df_cached.empty:
                print(f"⚠️  Using cached data due to API error")
                return df_cached
            return pd.DataFrame()
        
        resp_json = resp.json()
        
        if 'execution_id' not in resp_json:
            print(f"❌ No execution_id in response")
            if not df_cached.empty:
                return df_cached
            return pd.DataFrame()
        
        eid = resp_json["execution_id"]
        
        # Poll for completion with key rotation
        print(f"⏳ Waiting for query execution {eid[:8]}...", end="")
        for attempt in range(120):
            status_url = f"https://api.dune.com/api/v1/execution/{eid}/status"
            
            try:
                status_resp = dune_client.get(status_url, timeout=30)
                
                if status_resp.status_code != 200:
                    print(f"\n❌ Status check failed: {status_resp.status_code}")
                    break
                
                status_data = status_resp.json()
                state = status_data.get("state", "UNKNOWN")
                
                if state == "QUERY_STATE_COMPLETED":
                    print(" ✅")
                    break
                elif state in ["QUERY_STATE_FAILED", "QUERY_STATE_CANCELLED", "QUERY_STATE_EXPIRED"]:
                    print(f"\n❌ Query {state.lower()}")
                    if not df_cached.empty:
                        return df_cached
                    return pd.DataFrame()
                elif attempt % 6 == 0:
                    print(".", end="", flush=True)
                
            except Exception as e:
                print(f"\n⚠️  Status check error: {e}")
                break
            
            # Vary the wait time to avoid pattern detection
            wait_time = 10 + (attempt % 3)  # 10-12 seconds
            time.sleep(wait_time)
        
        else:
            print(f"\n⚠️  Query timed out after 20 minutes")
            if not df_cached.empty:
                return df_cached
            return pd.DataFrame()
        
        # Get results with key rotation
        results_url = f"https://api.dune.com/api/v1/execution/{eid}/results"
        results_resp = dune_client.get(results_url, timeout=30)
        
        if results_resp.status_code != 200:
            print(f"❌ Failed to get results: {results_resp.status_code}")
            if not df_cached.empty:
                return df_cached
            return pd.DataFrame()
        
        results_data = results_resp.json()
        
        if 'result' not in results_data or 'rows' not in results_data['result']:
            print(f"❌ No results in response")
            if not df_cached.empty:
                return df_cached
            return pd.DataFrame()
        
        rows = results_data["result"]["rows"]
        
        if not rows:
            print(f"⚠️  No new data returned for {fetch_start.date()} to {fetch_end.date()}")
            
            # ✅ FIX: If initial fetch returned no data, try a wider date range
            if df_cached.empty and fetch_start == DUNE_START:
                print(f"⚠️  Initial fetch returned empty, trying alternative approach...")
                # Try fetching from a later date
                alternative_start = DUNE_START + pd.Timedelta(days=1)
                print(f"   Trying from {alternative_start.date()}...")
                # You could implement a recursive call here or adjust the logic
                # For now, we'll return empty and let the pipeline handle it
            
            if not df_cached.empty:
                update_cache_timestamp(cache_path, df_cached)
            return df_cached
        
        # Process new data
        df_new = pd.DataFrame(rows)
        print(f"📥 Received {len(df_new)} new rows")
        print(f"   New data columns: {list(df_new.columns)}")
        
        if 'block_date' in df_new.columns:
            df_new["block_date"] = pd.to_datetime(df_new["block_date"], utc=True)
        else:
            print(f"❌ New data missing 'block_date' column!")
            return df_cached
        
        # ✅ FIX: Check if we need to fill missing dates
        if df_cached.empty and not df_new.empty:
            # This is initial fetch
            min_date = df_new['block_date'].min()
            print(f"   Initial data starts from: {min_date.date()}")
            
            if min_date > DUNE_START:
                print(f"⚠️  Warning: Data starts from {min_date.date()}, not {DUNE_START.date()}")
                print(f"   This might be a query limitation")
        
        # Merge with cached data
        if not df_cached.empty:
            common_cols = list(set(df_cached.columns) & set(df_new.columns))
            print(f"   Common columns between cache and new: {common_cols}")
            
            if not common_cols:
                print(f"❌ No common columns between cache and new data!")
                return df_cached
            
            df_cached = df_cached[common_cols]
            df_new = df_new[common_cols]
            
            df_combined = pd.concat([df_cached, df_new], ignore_index=True)
            df_combined = df_combined.drop_duplicates(
                subset=['block_date'],
                keep='last'
            ).sort_values('block_date').reset_index(drop=True)
            
            print(f"📊 Combined dataset: {len(df_combined)} rows ({len(df_new)} new)")
        else:
            df_combined = df_new
            print(f"📊 New dataset: {len(df_combined)} rows")
        
        # Update cache
        update_cache_file(cache_path, df_combined)
        
        # Log key rotation stats
        stats = key_rotator.get_stats()
        print(f"🔑 Key Rotation Stats: {stats['total_requests']} total requests")
        
        return df_combined
        
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        import traceback
        traceback.print_exc()
        return df_cached
def fetch_cg_chunked(cg_id, start, end, key, days=30):
    """Fetch CoinGecko prices in chunks"""
    url = "https://pro-api.coingecko.com/api/v3"
    headers = {"x-cg-pro-api-key": key}
    start_dt, end_dt = to_utc(start), to_utc(end) + pd.Timedelta(days=1)
    all_prices, curr = [], start_dt
    
    while curr < end_dt:
        next_dt = min(curr + pd.Timedelta(days=days), end_dt)
        params = {"vs_currency": "usd", "from": int(curr.timestamp()), 
                 "to": int(next_dt.timestamp())}
        r = requests.get(f"{url}/coins/{cg_id}/market_chart/range", 
                        params=params, headers=headers, timeout=30)
        all_prices.extend(r.json().get("prices", []))
        time.sleep(0.3)  # Be nice to CoinGecko API
        curr = next_dt
    
    df = pd.DataFrame(all_prices, columns=["timestamp", "price"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.floor("D")
    return df.groupby("date")["price"].mean().reset_index()

def update_cache_file(cache_path, df):
    """Helper to update cache file"""
    try:
        if 'block_date' in df.columns:
            df_dates = df['block_date'].copy()
            
            df_serializable = df.copy()
            df_serializable['block_date'] = df_serializable['block_date'].dt.strftime('%Y-%m-%d')
            
            with open(cache_path, 'w') as f:
                json.dump({
                    "last_block_date": df_dates.max().strftime("%Y-%m-%d"),
                    "data": json.loads(df_serializable.to_json(orient="records", date_format='iso'))
                }, f, indent=2)
            
            print(f"💾 Cache updated to {df_dates.max().date()}")
        else:
            print(f"⚠️  No block_date column in dataframe")
            
    except Exception as e:
        print(f"❌ Failed to update cache: {e}")

def update_cache_timestamp(cache_path, df_cached):
    """Update just the timestamp in cache without new data"""
    try:
        if os.path.exists(cache_path) and not df_cached.empty:
            with open(cache_path, 'r') as f:
                cache_data = json.load(f)
            
            yesterday = (pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
            cache_data["last_block_date"] = yesterday
            
            with open(cache_path, 'w') as f:
                json.dump(cache_data, f, indent=2)
            
            print(f"⏱️  Cache timestamp updated to {yesterday}")
    except Exception as e:
        print(f"⚠️  Could not update cache timestamp: {e}")

def get_price_incremental(symbol, cg_id, start, end, key):
    """Improved incremental price fetching"""
    cache_path = f"data/price_cache/{symbol}.csv"
    today_utc = pd.Timestamp.utcnow().floor("D")
    yesterday = today_utc - pd.Timedelta(days=1)
    start_dt, end_dt = to_utc(start), min(to_utc(end), yesterday)
    
    print(f"\n💰 Fetching {symbol.upper()} prices...")
    
    # Load existing cache
    if os.path.exists(cache_path):
        try:
            df = pd.read_csv(cache_path, parse_dates=["date"])
            df["date"] = df["date"].apply(to_utc)
            
            if not df.empty:
                last_date = df["date"].max()
                
                if last_date >= end_dt:
                    print(f"✅ {symbol.upper()} cache current ({last_date.date()})")
                    return df
                
                print(f"📅 {symbol.upper()} cache exists up to {last_date.date()}")
                fetch_start = last_date + pd.Timedelta(days=1)
            else:
                print(f"⚠️  {symbol.upper()} cache empty, fetching from {start_dt.date()}")
                fetch_start = start_dt
                
        except Exception as e:
            print(f"⚠️  {symbol.upper()} cache error: {e}, refetching...")
            fetch_start = start_dt
    else:
        print(f"📝 No {symbol.upper()} cache, fetching from {start_dt.date()}")
        fetch_start = start_dt
    
    # Check if we need to fetch
    if fetch_start > end_dt:
        print(f"✅ {symbol.upper()}: No new data needed")
        return df if 'df' in locals() else pd.DataFrame()
    
    print(f"🔍 {symbol.upper()}: Fetching {fetch_start.date()} to {end_dt.date()}")
    
    # Fetch new data
    try:
        new_data = fetch_cg_chunked(cg_id, fetch_start, end_dt, key)
        
        if new_data.empty:
            print(f"⚠️  {symbol.upper()}: No new price data returned")
            if 'df' in locals() and not df.empty:
                df.to_csv(cache_path, index=False)
                print(f"💾 {symbol.upper()} cache maintained")
            return df if 'df' in locals() else pd.DataFrame()
        
        # Merge with existing data
        if 'df' in locals() and not df.empty:
            new_data_dates = set(new_data['date'])
            existing_dates = set(df['date'])
            overlap = len(new_data_dates & existing_dates)
            
            if overlap > 0:
                print(f"⚠️  {symbol.upper()}: {overlap} overlapping dates, removing duplicates")
                df = df[~df['date'].isin(new_data_dates)]
            
            df_combined = pd.concat([df, new_data.rename(columns={"price": f"{symbol}_price"})], 
                                   ignore_index=True)
            df_combined = df_combined.drop_duplicates("date") \
                                     .sort_values("date") \
                                     .reset_index(drop=True)
            
            print(f"📊 {symbol.upper()}: Combined {len(new_data)} new, {len(df_combined)} total")
        else:
            df_combined = new_data.rename(columns={"price": f"{symbol}_price"})
            print(f"📊 {symbol.upper()}: {len(df_combined)} new rows")
        
        # Save to cache
        df_combined.to_csv(cache_path, index=False)
        print(f"💾 {symbol.upper()} cache updated to {df_combined['date'].max().date()}")
        
        return df_combined
        
    except Exception as e:
        print(f"❌ {symbol.upper()} fetch failed: {e}")
        if 'df' in locals():
            return df
        return pd.DataFrame()

def load_all_data_incremental():
    """Load all data sources with true incremental updates"""
    print("📊 Loading data sources incrementally...")
    
    # Fetch Dune data incrementally with key rotation
    df_whales = fetch_dune_incremental(
        QUERIES["whales"][0], 
        QUERIES["whales"][1],
        query_name="whale_data"
    )
    df_whales.to_csv(QUERIES["whales"][2], index=False)
    
    # Add delay between requests to avoid rate limiting
    time.sleep(2)  # Increased delay
    
    df_market = fetch_dune_incremental(
        QUERIES["market_intent"][0], 
        QUERIES["market_intent"][1],
        query_name="market_intent"
    )
    df_market.to_csv(QUERIES["market_intent"][2], index=False)
    
    # Determine date range for prices
    max_date = max(
        df_whales["block_date"].max() if not df_whales.empty else DUNE_START,
        df_market["block_date"].max() if not df_market.empty else DUNE_START
    )
    
    # Fetch prices incrementally
    df_btc = get_price_incremental("btc", "bitcoin", COINGECKO_START, max_date, COINGECKO_API_KEY)
    df_eth = get_price_incremental("eth", "ethereum", COINGECKO_START, max_date, COINGECKO_API_KEY)
    
    # Data quality check
    print(f"\n📈 Data Summary:")
    print(f"   Whale data:      {len(df_whales)} rows, {df_whales['block_date'].min().date() if not df_whales.empty else 'N/A'} to {df_whales['block_date'].max().date() if not df_whales.empty else 'N/A'}")
    print(f"   Market data:     {len(df_market)} rows, {df_market['block_date'].min().date() if not df_market.empty else 'N/A'} to {df_market['block_date'].max().date() if not df_market.empty else 'N/A'}")
    print(f"   BTC price:       {len(df_btc)} rows, {df_btc['date'].min().date() if not df_btc.empty else 'N/A'} to {df_btc['date'].max().date() if not df_btc.empty else 'N/A'}")
    print(f"   ETH price:       {len(df_eth)} rows, {df_eth['date'].min().date() if not df_eth.empty else 'N/A'} to {df_eth['date'].max().date() if not df_eth.empty else 'N/A'}")
    
    # Print key rotation stats
    stats = key_rotator.get_stats()
    print(f"\n🔑 Key Rotation Statistics:")
    print(f"   Total requests: {stats['total_requests']}")
    print(f"   Active keys: {stats['active_keys']}/{len(DUNE_API_KEYS)}")
    
    return df_whales, df_market, df_btc, df_eth

# ========== FEATURE ENGINEERING ==========
def add_price_features(df, price_col, prefix):
    """Add technical features for a price series"""
    df = df.copy()
    
    # Log returns
    df[f'{prefix}_log_return'] = np.log(df[price_col] / df[price_col].shift(1))
    
    # Lagged returns
    for lag in [1, 3, 7]:
        df[f'{prefix}_ret_lag{lag}'] = df[f'{prefix}_log_return'].shift(lag)
    
    # Volatility
    df[f'{prefix}_vol7'] = df[f'{prefix}_log_return'].rolling(7).std().shift(1)
    df[f'{prefix}_vol30'] = df[f'{prefix}_log_return'].rolling(30).std().shift(1)
    
    # RSI
    returns = df[f'{prefix}_log_return']
    gains = returns.where(returns > 0, 0).rolling(14).mean()
    losses = -returns.where(returns < 0, 0).rolling(14).mean()
    df[f'{prefix}_rsi'] = (100 - (100 / (1 + gains / (losses + 1e-10)))).shift(1)
    
    return df

def engineer_features(df_whales, df_market_intent, df_btc, df_eth):
    """Engineer all features with FIX A applied"""
    print("🔧 Engineering features...")
    
    # Merge price data
    df_prices = pd.merge(df_btc, df_eth, on='date', how='outer').sort_values('date')
    
    # Merge with whale data
    df = pd.merge(
        df_whales, 
        df_prices, 
        left_on='block_date', 
        right_on='date', 
        how='left'
    ).drop(columns=['date'])
    
    # Merge with market intent data
    df = pd.merge(
        df, 
        df_market_intent, 
        on='block_date', 
        how='left', 
        suffixes=('', '_intent')
    )
    
    df = df.sort_values('block_date').reset_index(drop=True)
    
    # Add price features
    df = add_price_features(df, 'eth_price', 'eth')
    df = add_price_features(df, 'btc_price', 'btc')
    
    # ETH/BTC ratio features
    df['eth_btc_ratio'] = df['eth_price'] / df['btc_price']
    df['eth_btc_ratio_ma7'] = df['eth_btc_ratio'].rolling(7).mean().shift(1)
    df['eth_btc_corr_30d'] = df['eth_log_return'].shift(1).rolling(30) \
        .corr(df['btc_log_return'].shift(1)).shift(1)
    
    # ✅ FIX A: Apply safe rolling z-scores
    zscore_pairs = [
        ('whale_tx_count', 'whale_tx_zscore_90d'),
        ('tx_per_active', 'tx_per_active_zscore_90d'),
        ('eth_burned', 'eth_burned_zscore_90d'),
        ('exchange_volume', 'exchange_volume_zscore'),
    ]
    
    for raw_col, zscore_col in zscore_pairs:
        if raw_col in df.columns:
            df[zscore_col] = rolling_zscore_safe(df[raw_col], 90)
    
    # Burn/issuance ratio
    if all(col in df.columns for col in ['eth_burned', 'total_gas_fees']):
        df['burn_issuance_ratio'] = (df['eth_burned'] / (df['total_gas_fees'] + 1e-10)).shift(1)
    
    # Whale volume deltas
    if 'whale_volume_ratio' in df.columns:
        df['whale_volume_ratio_delta_1d'] = df['whale_volume_ratio'].diff(1).shift(1)
        df['whale_volume_ratio_delta_3d'] = df['whale_volume_ratio'].diff(3).shift(1)
    
    # Clean up intermediate columns
    df = df.drop(columns=['eth_log_return', 'btc_log_return'], errors='ignore')
    
    # Save engineered features
    df.to_csv('data/features_engineered.csv', index=False)
    print(f"✅ Features engineered: {len(df.columns)} columns, {len(df)} rows")
    
    return df

# ========== TARGET CREATION ==========
def create_targets_two_tier(df, k=1.5):
    """Create two-tier SHORT labels (crash + breakdown)"""
    print("🎯 Creating two-tier targets...")
    
    df = df.sort_values('block_date').reset_index(drop=True).copy()
    
    # Calculate returns
    df['eth_log_return'] = np.log(df['eth_price'] / df['eth_price'].shift(1))
    df['rolling_vol_30'] = df['eth_log_return'].rolling(30, min_periods=10).std()
    
    # T+2 returns and threshold
    df['return_t2'] = df['eth_log_return'].rolling(2).sum().shift(-2)
    df['threshold_t2'] = df['rolling_vol_30'].rolling(30, min_periods=10).median() * k
    
    # Tier 1: Crash (hard down)
    hard_down = (
        (df['return_t2'] < -df['threshold_t2']) &
        (df['eth_vol7'] > df['eth_vol30']).fillna(False)
    )
    
    # Tier 2: Breakdown (pre-crash)
    exchange_flow_median = df['exchange_flow_share'].rolling(90, min_periods=30).median()
    
    soft_down = (
        (df['eth_ret_lag1'].fillna(0) < 0) &
        (df['btc_ret_lag1'].fillna(0) < 0) &
        (df['whale_volume_ratio_delta_3d'].fillna(0) > 0) &
        (df['exchange_flow_share'] > exchange_flow_median).fillna(False)
    )
    
    # Create targets
    df['target_t2'] = 0
    df.loc[df['return_t2'] > df['threshold_t2'], 'target_t2'] = 1  # UP
    df.loc[hard_down | soft_down, 'target_t2'] = -1  # DOWN (both tiers)
    
    # Create binary targets
    df['y_long_t2'] = (df['target_t2'] == 1).astype(int)
    df['y_short_t2'] = (df['target_t2'] == -1).astype(int)
    
    # Clean up
    df = df.drop(columns=['eth_log_return'], errors='ignore')
    
    # Print distribution
    print("\n📊 Target Distribution (Two-Tier SHORT):")
    for state, label in [(-1, 'DOWN'), (0, 'FLAT'), (1, 'UP')]:
        count = (df['target_t2'] == state).sum()
        percentage = count / len(df) * 100
        print(f"  {label:5s}: {count:4d} ({percentage:5.1f}%)")
    
    hard_count = hard_down.sum()
    soft_count = soft_down.sum()
    total_down = (df['target_t2'] == -1).sum()
    
    print(f"\n  Tier 1 (crash):     {hard_count:4d}")
    print(f"  Tier 2 (breakdown): {soft_count:4d}")
    print(f"  Total DOWN:         {total_down:4d}")
    
    return df

# ========== REGIME DEFINITION ==========
def define_regimes_extended(df):
    """Define trading regimes including R5 distribution regime"""
    print("📈 Defining extended regimes...")
    
    if 'btc_ret_lag1' not in df.columns or 'eth_vol7' not in df.columns:
        df['regime_code'] = 'R0'
        return df
    
    # Standard regimes based on BTC trend and ETH volatility
    btc_trend_7d = df['btc_ret_lag1'].rolling(7).mean()
    df['btc_regime'] = pd.cut(
        btc_trend_7d, 
        bins=[-np.inf, -0.005, 0.005, np.inf], 
        labels=['DOWN', 'FLAT', 'UP']
    )
    
    vol_median = df['eth_vol7'].rolling(180, min_periods=60).median()
    df['vol_regime'] = (df['eth_vol7'] > vol_median).map({True: 'HIGH', False: 'LOW'})
    
    # Combine for standard regimes
    df['regime'] = df['btc_regime'].astype(str) + '_' + df['vol_regime'].astype(str)
    regime_map = {
        'UP_HIGH': 'R1',    # Bull high vol
        'UP_LOW': 'R2',     # Bull low vol
        'DOWN_HIGH': 'R3',  # Bear high vol
        'DOWN_LOW': 'R4',   # Bear low vol
    }
    df['regime_code'] = df['regime'].map(regime_map).fillna('R0')
    
    # R5: Whale distribution regime
    exchange_flow_median = df['exchange_flow_share'].rolling(60, min_periods=20).median()
    
    df['dist_regime'] = (
        (df['whale_volume_ratio_delta_3d'].fillna(0) > 0) &
        (df['exchange_flow_share'] > exchange_flow_median).fillna(False)
    )
    
    # Override with R5 where distribution regime is active
    df.loc[df['dist_regime'], 'regime_code'] = 'R5'
    
    # Print regime distribution
    print("\n📊 Extended Regime Distribution:")
    regime_stats = []
    for code in ['R1', 'R2', 'R3', 'R4', 'R5', 'R0']:
        count = (df['regime_code'] == code).sum()
        if len(df) > 0:
            pct = count / len(df) * 100
            icon = '🟢' if code == 'R1' else ('🔴' if code in ['R3', 'R5'] else '⚪')
            regime_stats.append(f"{icon} {code}: {count:4d} ({pct:5.1f}%)")
    
    # Print in two columns
    for i in range(0, len(regime_stats), 2):
        row = regime_stats[i:i+2]
        print("  " + " | ".join(row))
    
    return df

# ========== BUILD COMPLETE PIPELINE ==========
def build_pipeline_complete(df_features):
    """
    Create the complete pipeline dataset with features, targets, and regimes
    """
    print("\n" + "="*70)
    print("BUILDING COMPLETE PIPELINE DATASET")
    print("="*70)
    
    # Create targets
    df_with_targets = create_targets_two_tier(df_features)
    
    # Define regimes
    df_complete = define_regimes_extended(df_with_targets)
    
    # Fill NaN values for features
    feature_cols = [col for col in df_complete.columns if col not in 
                   ['block_date', 'target_t2', 'y_long_t2', 'y_short_t2', 
                    'regime_code', 'btc_regime', 'vol_regime', 'regime', 'dist_regime']]
    
    df_complete[feature_cols] = df_complete[feature_cols].fillna(method='ffill').fillna(0)
    
    # Save complete pipeline
    df_complete.to_csv('data/pipeline_complete.csv', index=False)
    
    # Report statistics
    print(f"\n✅ Pipeline complete saved:")
    print(f"   Rows: {len(df_complete)}")
    print(f"   Columns: {len(df_complete.columns)}")
    print(f"   Date range: {df_complete['block_date'].min().date()} to {df_complete['block_date'].max().date()}")
    print(f"   File: data/pipeline_complete.csv")
    
    return df_complete

# ========== FIX 3: REDEFINED R3 SHORT PHILOSOPHY ==========
def check_r3_short_allowed(row):
    """
    ✅ FIX 3: Redefined R3 SHORT philosophy (early weakness only)
    R3 should be early weakness, not capitulation
    """
    # Small red, not dump
    small_red = (-0.015 < row['eth_ret_lag1'] < 0) if 'eth_ret_lag1' in row else False
    
    # BTC weakening
    btc_weak = (row['btc_ret_lag3'] < 0) if 'btc_ret_lag3' in row else False
    
    # NOT vol expansion (early, not panic)
    no_vol_expansion = False
    if 'eth_vol7' in row and 'eth_vol30' in row:
        no_vol_expansion = (row['eth_vol7'] <= row['eth_vol30'])
    
    # Whales increasing activity
    whale_activity = (row['whale_volume_ratio_delta_3d'] > 0) if 'whale_volume_ratio_delta_3d' in row else False
    
    return small_red and btc_weak and no_vol_expansion and whale_activity

# ========== FIX B, C: VETO SCORING ==========
def calculate_veto_score(row):
    """Calculate veto scores with proper classification"""
    veto = 0
    reasons = []
    
    structural_score = 0
    flow_score = 0
    context_score = 0
    
    # ✅ FIX B: Remove veto dominance - only count if liquidity exists
    if row.get('net_exchange_flow_ratio', 0) < 0 and row.get('exchange_volume_zscore', 0) > 0:
        veto += 1
        flow_score += 1
        reasons.append('net_flow_negative_with_liquidity')
    
    # STRUCTURAL VETOES
    if row.get('btc_ret_lag1', 0) < -0.02 and row.get('eth_ret_lag1', 0) < -0.01:
        veto += 2
        structural_score += 2
        reasons.append('btc_breakdown')
    
    if row.get('eth_vol7', 0) > row.get('eth_vol30', 0):
        veto += 2
        structural_score += 2
        reasons.append('vol_expansion')
    
    # FLOW VETOES
    if row.get('whale_exchange_flow_ratio', 0) > 0.6:
        veto += 1
        flow_score += 1
        reasons.append('whale_to_exchange')
    
    # CONTEXT VETOES
    if row.get('btc_ret_lag1', 0) > 0.02:
        veto += 1
        context_score += 1
        reasons.append('btc_conflict')
    
    if row.get('eth_vol7', 0) < row.get('eth_vol30', 0) * 0.7:
        veto += 1
        context_score += 1
        reasons.append('low_volatility')
    
    return veto, structural_score, flow_score, context_score, reasons

# ========== FIX E: CONFIDENCE SATURATION ==========
def adjust_confidence_with_saturation(prob, veto_score, regime):
    """Apply tanh saturation to prevent confidence runaway"""
    veto_boost = np.tanh(veto_score / 3) * 0.15
    adj_conf = np.clip(prob + veto_boost, 0, 0.95)
    
    # ✅ FIX 4: Cap R3 confidence at 0.70
    if regime == "R3":
        adj_conf = min(adj_conf, 0.70)
    
    return adj_conf

# ========== FIX F: FINAL SIGNAL OBJECT ==========
def build_final_signal_object(row, model_prob, regime, df=None):
    """Build the final signal object with all fixes applied"""
    
    # ✅ FIX 1: Price-not-near-lows requirement
    if df is not None:
        if not price_not_near_lows(row, df):
            return {
                "date": str(row['block_date'].date()),
                "regime": regime,
                "direction": None,
                "model_probability": float(model_prob),
                "structural_score": 0,
                "flow_score": 0,
                "context_score": 0,
                "adjusted_confidence": float(model_prob),
                "position_size": 0.0,
                "reasons": ["price_too_low"],
                "action": "NO_TRADE"
            }
    
    # Calculate veto scores
    veto, structural_score, flow_score, context_score, reasons = calculate_veto_score(row)
    
    # ✅ FIX 2: Flow required for all shorts
    if flow_score == 0:
        return {
            "date": str(row['block_date'].date()),
            "regime": regime,
            "direction": None,
            "model_probability": float(model_prob),
            "structural_score": int(structural_score),
            "flow_score": int(flow_score),
            "context_score": int(context_score),
            "adjusted_confidence": float(model_prob),
            "position_size": 0.0,
            "reasons": ["no_flow_confirmation"],
            "action": "NO_TRADE"
        }
    
    # ✅ FIX 5: R5 stronger flow requirement
    if regime == "R5" and flow_score < 2:
        return {
            "date": str(row['block_date'].date()),
            "regime": regime,
            "direction": None,
            "model_probability": float(model_prob),
            "structural_score": int(structural_score),
            "flow_score": int(flow_score),
            "context_score": int(context_score),
            "adjusted_confidence": float(model_prob),
            "position_size": 0.0,
            "reasons": ["weak_distribution_flow"],
            "action": "NO_TRADE"
        }
    
    # ✅ FIX C: Structural check - no structural weakness = no short
    if structural_score == 0:
        return {
            "date": str(row['block_date'].date()),
            "regime": regime,
            "direction": None,
            "model_probability": float(model_prob),
            "structural_score": int(structural_score),
            "flow_score": int(flow_score),
            "context_score": int(context_score),
            "adjusted_confidence": float(model_prob),
            "position_size": 0.0,
            "reasons": ["no_structural_break"],
            "action": "NO_TRADE"
        }
    
    # ✅ FIX 3: R3 short check (using new philosophy)
    if regime == "R3" and not check_r3_short_allowed(row):
        return {
            "date": str(row['block_date'].date()),
            "regime": regime,
            "direction": None,
            "model_probability": float(model_prob),
            "structural_score": int(structural_score),
            "flow_score": int(flow_score),
            "context_score": int(context_score),
            "adjusted_confidence": float(model_prob),
            "position_size": 0.0,
            "reasons": ["r3_no_early_weakness"],
            "action": "NO_TRADE"
        }
    
    # ✅ FIX E: Apply confidence saturation with regime-specific caps
    adjusted_confidence = adjust_confidence_with_saturation(model_prob, veto, regime)
    
    # Determine direction based on confidence
    if adjusted_confidence < 0.55:
        direction = None
        action = "NO_TRADE"
    else:
        direction = "SHORT"
        action = "ENTER"
    
    # ✅ FIX F: Position sizing
    position_size = map_confidence_to_size(adjusted_confidence)
    
    # Build final signal object
    signal = {
        "date": str(row['block_date'].date()),
        "regime": regime,
        "direction": direction,
        "model_probability": float(model_prob),
        "structural_score": int(structural_score),
        "flow_score": int(flow_score),
        "context_score": int(context_score),
        "adjusted_confidence": float(adjusted_confidence),
        "position_size": float(position_size),
        "reasons": reasons,
        "action": action
    }
    
    return signal

# ========== PHASE 1: WALK-FORWARD VALIDATION ==========
def walk_forward_r5_short(df):
    """PHASE 1: Walk-forward validation for R5 SHORT"""
    print("\n" + "="*70)
    print("PHASE 1: WALK-FORWARD VALIDATION (R5 SHORT ISOLATION)")
    print("="*70)
    
    splits = [
        {'train_start':'2017-01-01','train_end':'2020-12-31','test_start':'2021-01-01','test_end':'2021-12-31','name':'2021'},
        {'train_start':'2017-01-01','train_end':'2021-12-31','test_start':'2022-01-01','test_end':'2022-12-31','name':'2022'},
        {'train_start':'2017-01-01','train_end':'2022-12-31','test_start':'2023-01-01','test_end':'2023-12-31','name':'2023'},
        {'train_start':'2017-01-01','train_end':'2023-12-31','test_start':'2024-01-01','test_end':'2024-12-31','name':'2024'},
    ]
    
    results = []
    
    for split in splits:
        # Filter data for split
        train_mask = (df['block_date'] >= split['train_start']) & (df['block_date'] <= split['train_end'])
        test_mask = (df['block_date'] >= split['test_start']) & (df['block_date'] <= split['test_end'])
        
        train = df[train_mask].copy()
        test = df[test_mask].copy()
        
        # Filter to R5 regime only
        train_r5 = train[train['regime_code'] == 'R5'].copy()
        test_r5 = test[test['regime_code'] == 'R5'].copy()
        
        if len(train_r5) < 30 or len(test_r5) < 10:
            print(f"⚠️  {split['name']}: Insufficient R5 data")
            continue
        
        # Prepare features
        features = [f for f in SHORT_FEATURES if f in train_r5.columns]
        X_train = train_r5[features].fillna(method='ffill').fillna(0)
        y_train = train_r5['y_short_t2']
        X_test = test_r5[features].fillna(method='ffill').fillna(0)
        y_test = test_r5['y_short_t2']
        
        # Train model
        model = GradientBoostingClassifier(
            n_estimators=150,
            max_depth=4,
            learning_rate=0.05,
            random_state=42
        )
        model.fit(X_train, y_train)
        
        # Predict
        probs = model.predict_proba(X_test)[:, 1]
        preds = (probs > 0.55).astype(int)
        
        # Calculate metrics
        prec = precision_score(y_test, preds, zero_division=0)
        rec = recall_score(y_test, preds, zero_division=0)
        
        # Calculate AUC if we have both classes
        if len(np.unique(y_test)) > 1:
            auc = roc_auc_score(y_test, probs)
        else:
            auc = 0.0
        
        results.append({
            'period': split['name'],
            'precision': prec,
            'recall': rec,
            'auc': auc,
            'train_size': len(train_r5),
            'test_size': len(test_r5),
            'signals': preds.sum(),
            'threshold': 0.55
        })
        
        # Print results for this split
        status = "✅ PASS" if prec >= 0.65 and rec >= 0.50 else "❌ FAIL"
        print(f"\n{split['name']} {status}")
        print(f"  Precision: {prec:.3f} (req: ≥0.65)")
        print(f"  Recall:    {rec:.3f} (req: ≥0.50)")
        print(f"  AUC:       {auc:.3f}")
        print(f"  Signals:   {preds.sum()}/{len(test_r5)}")
        print(f"  R5 days:   {len(test_r5)}")
    
    # Save results
    df_results = pd.DataFrame(results)
    df_results.to_csv('validation/walk_forward_r5.csv', index=False)
    
    # Calculate overall verdict
    passed = ((df_results['precision'] >= 0.65) & (df_results['recall'] >= 0.50)).sum()
    total = len(df_results)
    
    print(f"\n{'='*70}")
    print(f"VERDICT: {passed}/{total} periods passed")
    if passed == total:
        print("✅ R5 SHORT APPROVED FOR DEPLOYMENT")
    elif passed >= total * 0.75:
        print("⚠️  R5 SHORT CONDITIONALLY APPROVED (reduce position size)")
    else:
        print("❌ R5 SHORT FAILED (do not deploy)")
    print(f"{'='*70}")
    
    return df_results

# ========== PHASE 2: PNL SIMULATION ==========
def simulate_pnl(df):
    """PHASE 2: Regime-aware PnL simulation"""
    print("\n" + "="*70)
    print("PHASE 2: REGIME-AWARE PNL SIMULATION")
    print("="*70)
    
    df = df.sort_values('block_date').reset_index(drop=True).copy()
    
    # Check if we have enough data for T+2
    if len(df) < 3:
        print("⚠️  Insufficient data for PnL simulation")
        return pd.DataFrame()
    
    # Calculate future price for exit (only for rows where we can calculate T+2)
    df['price_t2'] = np.nan
    # Only calculate for rows where we have enough future data
    for i in range(len(df) - 2):
        df.loc[i, 'price_t2'] = df.loc[i + 2, 'eth_price']
    
    # Filter to R5 regime for training
    df_r5 = df[df['regime_code'] == 'R5'].copy()
    
    if len(df_r5) < 50:
        print("⚠️  Insufficient R5 data for PnL simulation")
        return pd.DataFrame()
    
    # Prepare features
    features = [f for f in SHORT_FEATURES if f in df_r5.columns]
    
    # Train-test split (80/20)
    split_idx = int(len(df_r5) * 0.8)
    X_train = df_r5[features].iloc[:split_idx].fillna(method='ffill').fillna(0)
    y_train = df_r5['y_short_t2'].iloc[:split_idx]
    
    # Train model
    model = GradientBoostingClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.05,
        random_state=42
    )
    model.fit(X_train, y_train)
    
    # Generate signals for entire dataset
    X_all = df[features].fillna(method='ffill').fillna(0)
    df['signal_prob'] = model.predict_proba(X_all)[:, 1]
    df['signal_raw'] = ((df['signal_prob'] > 0.55) & (df['regime_code'] == 'R5')).astype(int)
    
    # Apply all fixes to generate final signals
    df['signal'] = 0
    df['signal_reason'] = ""
    
    for i, row in df.iterrows():
        if row['signal_raw'] == 1:
            signal_obj = build_final_signal_object(row, row['signal_prob'], row['regime_code'], df)
            df.loc[i, 'signal'] = 1 if signal_obj['action'] == 'ENTER' else 0
            df.loc[i, 'signal_reason'] = ",".join(signal_obj['reasons'])
    
    # Simulate trades
    trades = []
    cooldown_until = None
    
    for i, row in df.iterrows():
        # Skip if we don't have exit price or entry price
        if pd.isna(row.get('price_t2')) or pd.isna(row.get('eth_price')):
            continue
        
        # Skip if price_t2 is NaN
        if pd.isna(row['price_t2']):
            continue
        
        # Check cooldown
        if cooldown_until and row['block_date'] <= cooldown_until:
            continue
        
        # Check signal with all fixes applied
        if row['signal'] == 1:
            # Calculate returns (SHORT position)
            gross_ret = -(row['price_t2'] - row['eth_price']) / row['eth_price']
            net_ret = gross_ret - (2 * SLIPPAGE) - (2 * FEES)
            
            # Get veto scores for this trade
            veto, structural_score, flow_score, context_score, reasons = calculate_veto_score(row)
            
            trades.append({
                'entry_date': row['block_date'],
                'exit_date': row['block_date'] + timedelta(days=2),
                'regime': row['regime_code'],
                'entry_price': row['eth_price'],
                'exit_price': row['price_t2'],
                'gross_ret': gross_ret,
                'net_ret': net_ret,
                'confidence': row['signal_prob'],
                'direction': 'SHORT',
                'structural_score': structural_score,
                'flow_score': flow_score,
                'context_score': context_score,
                'reason': row['signal_reason']
            })
            
            # Apply cooldown
            cooldown_until = row['block_date'] + timedelta(days=1)
    
    # Create trades DataFrame
    if not trades:
        print("⚠️  No trades generated in simulation")
        return pd.DataFrame()
    
    df_trades = pd.DataFrame(trades)
    
    # Calculate performance metrics
    print("\n📊 PnL Attribution by Regime:")
    if not df_trades.empty:
        regime_stats = df_trades.groupby('regime').agg({
            'net_ret': ['count', 'mean', 'sum', 'std', 'max', 'min']
        }).round(4)
        print(regime_stats)
    else:
        print("   No trades to analyze")
    
    # Overall performance
    total_trades = len(df_trades)
    total_return = df_trades['net_ret'].sum()
    win_rate = (df_trades['net_ret'] > 0).mean()
    avg_win = df_trades[df_trades['net_ret'] > 0]['net_ret'].mean() if win_rate > 0 else 0
    avg_loss = df_trades[df_trades['net_ret'] <= 0]['net_ret'].mean() if win_rate < 1 else 0
    
    # Sharpe ratio (annualized)
    if not df_trades.empty and df_trades['net_ret'].std() > 0:
        sharpe = df_trades['net_ret'].mean() / df_trades['net_ret'].std() * np.sqrt(252 / 3)
    else:
        sharpe = 0
    
    print(f"\n📈 Overall Performance:")
    print(f"  Total trades:      {total_trades}")
    print(f"  Total return:      {total_return * 100:.2f}%")
    print(f"  Win rate:          {win_rate * 100:.1f}%" if not pd.isna(win_rate) else "  Win rate:          N/A")
    print(f"  Average win:       {avg_win * 100:.2f}%" if avg_win != 0 else "  Average win:       N/A")
    print(f"  Average loss:      {avg_loss * 100:.2f}%" if avg_loss != 0 else "  Average loss:      N/A")
    print(f"  Sharpe (annual):   {sharpe:.2f}")
    if not df_trades.empty:
        print(f"  Max single win:    {df_trades['net_ret'].max() * 100:.2f}%")
        print(f"  Max single loss:   {df_trades['net_ret'].min() * 100:.2f}%")
    
    # Save trades
    df_trades.to_csv('backtest/pnl_simulation.csv', index=False)
    print(f"\n✅ PnL simulation saved: backtest/pnl_simulation.csv")
    
    return df_trades

# ========== PHASE 3-6: COMPLETE SIGNAL GENERATION ==========
def generate_daily_signal(df, model):
    """Generate daily trading signal with all fixes applied"""
    # Get latest data
    latest_row = df.iloc[-1].copy()
    
    # Get model probability
    features = [f for f in SHORT_FEATURES if f in df.columns]
    X_latest = latest_row[features].fillna(0).values.reshape(1, -1)
    model_prob = model.predict_proba(X_latest)[0, 1] if hasattr(model, 'predict_proba') else 0.5
    
    # Get regime
    regime = latest_row.get('regime_code', 'R0')
    
    # Build final signal object with all fixes
    signal = build_final_signal_object(latest_row, model_prob, regime, df)
    
    return signal

# ========== PHASE 5: STRESS TESTS ==========
def stress_tests(df, df_trades=None):
    """PHASE 5: Mandatory stress tests"""
    print("\n" + "="*70)
    print("PHASE 5: STRESS TESTS")
    print("="*70)
    
    # 1. Bull Market Stress (2020-2021)
    bull_period = df[(df['block_date'] >= '2020-01-01') & (df['block_date'] <= '2021-12-31')]
    bull_r5_days = (bull_period['regime_code'] == 'R5').sum()
    
    bull_trade_count = 0
    bull_pnl = 0
    
    if df_trades is not None and not df_trades.empty and 'entry_date' in df_trades.columns:
        bull_trades = df_trades[
            (df_trades['entry_date'] >= '2020-01-01') & 
            (df_trades['entry_date'] <= '2021-12-31')
        ]
        bull_trade_count = len(bull_trades)
        bull_pnl = bull_trades['net_ret'].sum() if not bull_trades.empty else 0
    
    print(f"\n1️⃣ Bull Market Stress (2020-2021):")
    print(f"   R5 days:          {bull_r5_days} (expect: low)")
    print(f"   Trades generated: {bull_trade_count} (expect: few)")
    print(f"   Total PnL:        {bull_pnl * 100:.2f}% (expect: low drawdown)")
    
    # 2. Crash Cluster Stress
    crash_periods = [
        ('Mar 2020', '2020-03-01', '2020-03-31'),
        ('Nov 2022', '2022-11-01', '2022-11-30')
    ]
    
    print(f"\n2️⃣ Crash Cluster Stress:")
    for name, start, end in crash_periods:
        crash_mask = (df['block_date'] >= start) & (df['block_date'] <= end)
        crash_r5 = df[crash_mask]
        r5_count = (crash_r5['regime_code'] == 'R5').sum()
        
        crash_pnl = 0
        trade_count = 0
        
        if df_trades is not None and not df_trades.empty and 'entry_date' in df_trades.columns:
            crash_trades = df_trades[
                (df_trades['entry_date'] >= start) & 
                (df_trades['entry_date'] <= end)
            ]
            crash_pnl = crash_trades['net_ret'].sum() if not crash_trades.empty else 0
            trade_count = len(crash_trades)
        
        print(f"   {name}:")
        print(f"     R5 days:     {r5_count}")
        print(f"     Trades:      {trade_count}")
        print(f"     PnL:         {crash_pnl * 100:.2f}% (expect: positive convexity)")
    
    # 3. Signal Density Check
    print(f"\n3️⃣ Signal Density Check:")
    
    if df_trades is not None and not df_trades.empty and 'entry_date' in df_trades.columns:
        try:
            # Convert entry_date to datetime if it's not already
            if not pd.api.types.is_datetime64_any_dtype(df_trades['entry_date']):
                df_trades['entry_date'] = pd.to_datetime(df_trades['entry_date'])
            
            df_trades['year'] = df_trades['entry_date'].dt.year
            yearly_trades = df_trades.groupby('year').size()
            
            print("   Yearly trade count:")
            for year, count in yearly_trades.items():
                status = "✅" if count <= 40 else "⚠️ "
                print(f"     {status} {year}: {count} trades")
            
            if yearly_trades.max() > 40:
                print("   ⚠️  WARNING: Overtrading detected")
            else:
                print("   ✅ Acceptable trade density")
        except Exception as e:
            print(f"   Error analyzing trade density: {e}")
    else:
        print("   No trades to analyze density")
    
    # 4. Fix Validation Check
    print(f"\n4️⃣ Fix Validation Check:")
    
    # Check if we have recent data to analyze
    recent_data = df[df['block_date'] >= '2024-01-01']
    if not recent_data.empty:
        # Count R3 vs R5 days
        r3_days = (recent_data['regime_code'] == 'R3').sum()
        r5_days = (recent_data['regime_code'] == 'R5').sum()
        
        print(f"   Recent R3 days: {r3_days}")
        print(f"   Recent R5 days: {r5_days}")
        
        # Check price distribution for potential shorts
        if 'eth_price' in recent_data.columns:
            recent_prices = recent_data['eth_price'].dropna()
            if len(recent_prices) > 0:
                price_min = recent_prices.min()
                price_max = recent_prices.max()
                current_price = recent_prices.iloc[-1] if len(recent_prices) > 0 else 0
                
                if price_max - price_min > 0:
                    current_percentile = (current_price - price_min) / (price_max - price_min)
                    print(f"   Current price percentile: {current_percentile:.2f}")
                    if current_percentile < 0.25:
                        print("   ⚠️  Current price in bottom 25% - shorts should be blocked")
                    else:
                        print("   ✅ Current price not near lows")
    
    print(f"\n{'='*70}")

# ========== PHASE 6: DEPLOYMENT CHECKLIST ==========
def deployment_checklist():
    """PHASE 6: Live deployment checklist"""
    print("\n" + "="*70)
    print("PHASE 6: LIVE DEPLOYMENT CHECKLIST")
    print("="*70)
    
    checks = {
        'Feature lag integrity': os.path.exists('data/features_engineered.csv'),
        'Data freshness': os.path.exists('data/dune_whales_cache.json'),
        'Model versioning': os.path.exists('models'),
        'Daily log structure': os.path.exists('data'),
        'Validation results': os.path.exists('validation/walk_forward_r5.csv'),
        'Backtest results': os.path.exists('backtest/pnl_simulation.csv'),
        'Complete pipeline': os.path.exists('data/pipeline_complete.csv'),
        'Key rotation logs': os.path.exists('logs')
    }
    
    print("✅ Automated checks:")
    for check, passed in checks.items():
        print(f"   {'✅' if passed else '❌'} {check}")
    
    print("\n📋 Manual checks required:")
    print("   ⚠️  API circuit breaker implementation")
    print("   ⚠️  Kill-switch logic (3 consecutive losses)")
    print("   ⚠️  Data pipeline monitoring")
    print("   ⚠️  Model retraining schedule")
    
    print(f"\n{'='*70}")

# ========== MAIN PIPELINE EXECUTION ==========
def run_complete_pipeline():
    """Execute the complete 6-phase pipeline"""
    print("\n" + "="*70)
    print("ETH WHALE ALPHA PIPELINE - COMPLETE EXECUTION (WITH FIXES)")
    print("="*70)
    
    # Step 1: Load data
    df_whales, df_market, df_btc, df_eth = load_all_data_incremental()
    
    # Step 2: Engineer features
    df_features = engineer_features(df_whales, df_market, df_btc, df_eth)
    
    # Step 3: Build complete pipeline
    df_pipeline = build_pipeline_complete(df_features)
    
    # Step 4: Train final R5 model for signal generation
    print("\n" + "="*70)
    print("TRAINING FINAL R5 SHORT MODEL")
    print("="*70)
    
    df_r5 = df_pipeline[df_pipeline['regime_code'] == 'R5'].copy()
    features = [f for f in SHORT_FEATURES if f in df_r5.columns]
    
    if len(df_r5) < 50:
        print("⚠️  Insufficient R5 data for model training")
        return None, None, None
    
    # Use 80% of data for training
    split_idx = int(len(df_r5) * 0.8)
    X_train = df_r5[features].iloc[:split_idx].fillna(method='ffill').fillna(0)
    y_train = df_r5['y_short_t2'].iloc[:split_idx]
    
    # Train final model
    final_model = GradientBoostingClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.05,
        random_state=42
    )
    final_model.fit(X_train, y_train)
    
    # Save model
    joblib.dump(final_model, 'models/r5_short_final.pkl')
    print(f"✅ Final model saved: models/r5_short_final.pkl")
    
    # Step 5: Execute all phases
    print("\n" + "="*70)
    print("EXECUTING ALL 6 PHASES")
    print("="*70)
    
    # Phase 1: Walk-forward validation
    wf_results = walk_forward_r5_short(df_pipeline)
    
    # Phase 2: PnL simulation
    trades = simulate_pnl(df_pipeline)
    
    # Phase 3-4: Generate final signal
    signal = generate_daily_signal(df_pipeline, final_model)
    
    print("\n" + "="*70)
    print("FINAL TRADING SIGNAL")
    print("="*70)
    print(json.dumps(signal, indent=2))
    
    # Phase 5: Stress tests - handle empty trades
    stress_tests(df_pipeline, trades if trades is not None and not trades.empty else None)
    
    # Phase 6: Deployment checklist
    deployment_checklist()
    
    # Save final signal
    with open('data/latest_signal.json', 'w') as f:
        json.dump(signal, f, indent=2)
    
    print(f"\n✅ Pipeline execution complete!")
    print(f"   Final signal saved: data/latest_signal.json")
    print(f"   Pipeline data: data/pipeline_complete.csv ({len(df_pipeline)} rows)")
    
    # Print key rotation stats
    stats = key_rotator.get_stats()
    print(f"\n🔑 Key Rotation Summary:")
    print(f"   Total requests: {stats['total_requests']}")
    print(f"   Active keys: {stats['active_keys']}/{len(DUNE_API_KEYS)}")
    
    return wf_results, trades, signal

# ========== PAPER TRADE TEST ==========
def run_paper_trade_test():
    """Run a 60-day paper trade test to validate fixes"""
    print("\n" + "="*70)
    print("60-DAY PAPER TRADE TEST (VALIDATING FIXES)")
    print("="*70)
    
    # Load pipeline data
    if not os.path.exists('data/pipeline_complete.csv'):
        print("❌ Pipeline data not found. Run pipeline first.")
        return
    
    df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
    df = df.sort_values('block_date')
    
    # Take last 60 days
    test_period = df.iloc[-60:].copy()
    
    # Generate signals for test period
    signals = []
    for i, row in test_period.iterrows():
        # Simulate model probability (in reality would use trained model)
        model_prob = 0.6 + np.random.uniform(-0.1, 0.1)
        
        # Get signal with all fixes applied
        signal = build_final_signal_object(row, model_prob, row['regime_code'], df)
        signals.append(signal)
        
        # Print trade signals
        if signal['action'] == 'ENTER':
            print(f"{signal['date']}: {signal['action']} {signal['direction']} "
                  f"@ {row['eth_price']:.0f} (conf: {signal['adjusted_confidence']:.2f}, "
                  f"size: {signal['position_size']}, flow_score: {signal['flow_score']})")
    
    # Analyze results
    df_signals = pd.DataFrame(signals)
    
    print(f"\n📊 Test Results (60 days):")
    print(f"   Total signals: {len(df_signals)}")
    print(f"   Enter signals: {(df_signals['action'] == 'ENTER').sum()}")
    
    # Count different rejection reasons
    if len(df_signals) > 0:
        # Initialize counters
        price_low_count = 0
        no_flow_count = 0
        weak_dist_flow_count = 0
        no_structure_count = 0
        r3_early_count = 0
        
        for signal in signals:
            if isinstance(signal.get('reasons'), list):
                reasons = signal['reasons']
                if 'price_too_low' in reasons:
                    price_low_count += 1
                if 'no_flow_confirmation' in reasons:
                    no_flow_count += 1
                if 'weak_distribution_flow' in reasons:
                    weak_dist_flow_count += 1
                if 'no_structural_break' in reasons:
                    no_structure_count += 1
                if 'r3_no_early_weakness' in reasons:
                    r3_early_count += 1
        
        print(f"   No-trade due to price_too_low: {price_low_count}")
        print(f"   No-trade due to no_flow_confirmation: {no_flow_count}")
        print(f"   No-trade due to weak_distribution_flow: {weak_dist_flow_count}")
        print(f"   No-trade due to no_structural_break: {no_structure_count}")
        print(f"   No-trade due to r3_no_early_weakness: {r3_early_count}")
    
    # Calculate average entry price percentile for enter signals
    enter_signals = df_signals[df_signals['action'] == 'ENTER']
    if not enter_signals.empty:
        entry_dates = pd.to_datetime(enter_signals['date'])
        entry_prices = []
        
        for date in entry_dates:
            # Find the price on this date
            price_row = df[df['block_date'] == date]
            if not price_row.empty:
                entry_prices.append(price_row.iloc[0]['eth_price'])
        
        # Calculate price percentiles for entry points
        price_percentiles = []
        for price in entry_prices:
            # Calculate percentile relative to last 90 days of the full dataframe
            recent_mask = (df['block_date'] >= (date - timedelta(days=90))) & (df['block_date'] < date)
            recent_prices = df[recent_mask]['eth_price'].values
            
            if len(recent_prices) > 0:
                percentile = np.sum(price > recent_prices) / len(recent_prices)
                price_percentiles.append(percentile)
        
        if price_percentiles:
            avg_percentile = np.mean(price_percentiles)
            print(f"   Average entry price percentile: {avg_percentile:.2f} (target: >0.25)")
            
            # Check if shorts are clustering near tops vs bottoms
            low_percentile_shorts = sum(1 for p in price_percentiles if p < 0.25)
            high_percentile_shorts = sum(1 for p in price_percentiles if p > 0.75)
            print(f"   Shorts in bottom 25%: {low_percentile_shorts} (target: 0)")
            print(f"   Shorts in top 25%: {high_percentile_shorts} (target: >0)")
    
    return df_signals

# ========== MAIN EXECUTION ==========
if __name__ == "__main__":
    print("\n" + "="*70)
    print("ETH WHALE ALPHA - SURGICAL FIXES APPLIED")
    print("="*70)
    
    # First test the connection
    connection_ok = test_dune_connection()
    
    if not connection_ok:
        print("\n⚠️  Dune connection test failed. Possible issues:")
        print("   1. DUNE_API_KEY not set in .env file")
        print("   2. Dune API key expired or invalid")
        print("   3. Network/firewall issues")
        print("   4. Dune query ID might be incorrect")
        print("\nPlease check your .env file and Dune API key.")
        exit(1)
            
    # Option 1: Run complete pipeline
    run_full = input("\nRun complete pipeline? (y/n): ").lower() == 'y'
    
    if run_full:
        wf_results, trades, final_signal = run_complete_pipeline()
    else:
        # Option 2: Just test the fixes with paper trades
        paper_results = run_paper_trade_test()
    
    # Summary
    if 'final_signal' in locals() and final_signal:
        print("\n" + "="*70)
        print("SUMMARY")
        print("="*70)
        print(f"Signal: {final_signal['action']}")
        print(f"Regime: {final_signal['regime']}")
        print(f"Confidence: {final_signal['adjusted_confidence']:.3f}")
        print(f"Position Size: {final_signal['position_size']}")
        print(f"Structural Score: {final_signal['structural_score']}")
        print(f"Flow Score: {final_signal['flow_score']}")
        print(f"Context Score: {final_signal['context_score']}")
        print(f"Reasons: {', '.join(final_signal['reasons'])}")
        print(f"{'='*70}")
        print("\n🎯 Expected changes after fixes:")
        print("   • Shorts will appear near local highs/mid-range")
        print("   • R3: rare, small, early entries")
        print("   • R5: fewer, larger, later entries")
        print("   • Flow features will be decisive")
        print("   • No more shorts at bottoms")